# Perturbation boundary conditions

Every single-port terminal carries a `PerturbationBC` — the *acoustic* closure the mean
boundary condition cannot supply. Theory (§12.4) reduces each one to a **characteristic
closure** mapping the waves that *arrive* at the boundary to the waves the boundary must
*specify*,

$$ w_\text{specify} = A(\omega)\, w_\text{arriving} + b(\omega). $$

For a subsonic terminal the scalar **reflection** $R$ sits on the acoustic↔acoustic entry of
$A$; off-diagonal entries carry entropy↔acoustic coupling, and $b$ is an excitation forcing.
This notebook drives a single duct and exercises **every** named closure, checking each
measured response against its analytic value.

| constructor | closure |
|---|---|
| `inherit` (default) | *no stamp* — keep the linearized mean BC |
| `hard_wall` | $R = +1$ ($u'=0$) |
| `open_end` | $R = -1$ ($p'=0$, pressure release) |
| `mean_flow_open_end` | $R = -(1-M)/(1+M)$ |
| `anechoic` | $R = 0$ |
| `reflection(R)` | user $R$ — constant / table / callable |
| `impedance(Z)` / `impedance_polar` | $R = (Z-\rho c)/(Z+\rho c)$ |
| `excitation(b)` | forcing $b$ on an incoming wave (the source) |
| `choked_nozzle` (alias `compact_nozzle`) | $g = Rf + R_s h$, Marble–Candel compact outlet |
| `constant_mass_flow` | $g = Rf + R_s h$, pins $\dot m' = 0$ |


In [ ]:
import os, sys

_root = os.getcwd()
while not os.path.isdir(os.path.join(_root, "nefes")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from nefes.elements import catalog as cat
from nefes.perturbation import forced_response
from nefes.perturbation.operator.boundary_bc import PerturbationBC
from nefes.plotting import use_nefes_theme
from nefes.solver import solve
from nefes.solver.control import states_table
from nefes.thermo.configure import perfect_gas
from nefes.assembly.derive import ES_C, ES_M, ES_RHO

use_nefes_theme()
R_AIR, GAMMA = 287.0, 1.4
CP = GAMMA * R_AIR / (GAMMA - 1.0)
L, AREA, OUT = 0.5, 0.05, 1  # duct length [m], area [m^2], outlet edge id

## A reusable terminated duct

`drv` is a clean acoustic source (`excitation`); a `pressure_outlet` carries the closure under
test. We read the perturbation waves $(f,g,h)$ at the outlet edge and report the mean state
there. Reading $g/f$ *right at the termination* recovers the stamped reflection $R$ with no
round-trip phase — the cleanest check of a boundary.

In [ ]:
def terminate(bc, inlet_bc=None, freqs=np.array([200.0])):
    """Solve an excitation-driven duct terminated by `bc`; return (ForcedResponse, mean dict)."""
    inlet_bc = inlet_bc or PerturbationBC.anechoic(driven=("acoustic",))
    els = [
        cat.total_pressure_inlet(108000.0, 300.0, name="drv", perturbation_bc=inlet_bc),
        cat.duct(L, name="duct"),
        cat.pressure_outlet(101325.0, 300.0, name="term", perturbation_bc=bc),
    ]
    prob = cat.build_problem(perfect_gas(R_AIR, GAMMA), els, [(0, 1, AREA), (1, 2, AREA)], 5.0, 1e5, CP * 300.0)
    res = solve(prob)
    assert res.converged
    est = states_table(prob, res.x)
    mean = dict(M=float(est[ES_M, OUT]), c=float(est[ES_C, OUT]), rho=float(est[ES_RHO, OUT]))
    return forced_response(prob, res.x, freqs), mean


_, mean = terminate(PerturbationBC.anechoic())
M, c, rho = mean["M"], mean["c"], mean["rho"]
print(f"outlet mean state:  M = {M:.4f},  c = {c:.1f} m/s,  rho = {rho:.3f} kg/m^3")

## 1. Diagonal reflection terminations

The six closures whose $A$ is a pure reflection $R$. Each is constant in frequency here (fixed
outlet $M$), so the measured $g/f$ is flat; we plot every $R$ in the complex plane against its
analytic value and confirm they coincide to round-off.

In [ ]:
zeta = 2.0 + 0.0j  # specific impedance for the impedance termination
diagonal = {
    "hard_wall": (PerturbationBC.hard_wall(), 1.0 + 0j),
    "open_end": (PerturbationBC.open_end(), -1.0 + 0j),
    "mean_flow_open_end": (PerturbationBC.mean_flow_open_end(), -(1 - M) / (1 + M)),
    "anechoic": (PerturbationBC.anechoic(), 0.0 + 0j),
    "reflection(0.4+0.3j)": (PerturbationBC.reflection(0.4 + 0.3j), 0.4 + 0.3j),
    "impedance_polar(|2|,0)": (PerturbationBC.impedance_polar(2.0, 0.0), (zeta - 1) / (zeta + 1)),
}

print(f"{'closure':24s} {'measured g/f':>22s} {'analytic R':>22s}")
meas, theo = {}, {}
for name, (bc, R) in diagonal.items():
    fr, _ = terminate(bc)
    meas[name] = fr.reflection_at(OUT)[0]
    theo[name] = complex(R)
    print(f"{name:24s} {meas[name]:>+22.4f} {theo[name]:>+22.4f}")
print(f"\nmax |measured - analytic| = {max(abs(meas[k] - theo[k]) for k in diagonal):.2e}")

In [ ]:
th = np.linspace(0, 2 * np.pi, 200)
fig = go.Figure()
fig.add_scatter(x=np.cos(th), y=np.sin(th), mode="lines", line=dict(color="gray", dash="dot"),
                name="|R| = 1", hoverinfo="skip")
fig.add_scatter(x=[v.real for v in theo.values()], y=[v.imag for v in theo.values()], mode="markers",
                marker=dict(symbol="x", size=13, color="black"), name="analytic")
fig.add_scatter(x=[v.real for v in meas.values()], y=[v.imag for v in meas.values()], mode="markers+text",
                marker=dict(symbol="circle-open", size=12), text=list(meas), textposition="top center",
                name="measured")
fig.update_layout(title="Reflection terminations in the complex R-plane",
                  xaxis_title="Re R", yaxis_title="Im R", yaxis_scaleanchor="x", yaxis_scaleratio=1)
fig.show()

Each measured `o` lands on its analytic `x`. Note `mean_flow_open_end` is pulled inside the unit
circle by the mean flow ($R=-(1-M)/(1+M)$, vs $-1$ for the ideal `open_end`), and the specific
impedance $\zeta=2$ maps to $R=(\zeta-1)/(\zeta+1)=1/3$.

## 2. Excitation — the source term $b$

`excitation` is the only closure that contributes the forcing $b$; without one the forced
problem has no drive. It seats an incoming wave of a chosen `family` (`"acoustic"` or
`"entropy"`) on top of an optional `base_R` (the source's own reflection, default $0$ — a clean
anechoic source). Below: a reflective source `base_R = 0.5` reflects returning waves while still
injecting unit amplitude.

In [ ]:
freqs = np.linspace(20.0, 1200.0, 200)
fr_clean, _ = terminate(PerturbationBC.hard_wall(), inlet_bc=PerturbationBC.anechoic(driven=("acoustic",)), freqs=freqs)
fr_refl, _ = terminate(PerturbationBC.hard_wall(),
                       inlet_bc=PerturbationBC.reflection(0.5, driven=("acoustic",)), freqs=freqs)

# |p'| at the inlet edge: a clean source is anechoic, a reflective source builds a standing wave.
p_clean = np.abs(fr_clean.field(0, "primitive")[:, 1])
p_refl = np.abs(fr_refl.field(0, "primitive")[:, 1])
fig = go.Figure()
fig.add_scatter(x=freqs, y=p_clean, mode="lines", name="base_R = 0 (clean)")
fig.add_scatter(x=freqs, y=p_refl, mode="lines", name="base_R = 0.5 (reflective)")
fig.update_layout(title="Excitation source: |p'| at the inlet vs source reflectivity",
                  xaxis_title="frequency [Hz]", yaxis_title="|p'|")
fig.show()

The reflective source closes a resonator with the hard-wall outlet, so its inlet pressure shows
the duct's standing-wave peaks; the clean source absorbs the return and stays smooth.

## 3. Entropy-coupling outlets — indirect noise

`choked_nozzle` (Marble–Candel compact outlet) and `constant_mass_flow` are **outlet**
terminations whose $A$ has an off-diagonal $R_s$: an arriving *entropy* wave reflects into an
outgoing *acoustic* wave ($g = Rf + R_s h$) — the mechanism of indirect/entropy noise. We drive
a unit **entropy** wave at the inlet, so $f\!\to\!0$ at the outlet and the measured $g/h$ isolates
$R_s$.

$$ \text{choked:}\;\; R_s = \frac{c}{\rho}\frac{M}{2+(\gamma-1)M}, \qquad
   \text{const.\ }\dot m:\;\; R_s = \frac{c\,M}{\rho(1-M)}. $$

In [ ]:
gm1 = GAMMA - 1.0
couplers = {
    "choked_nozzle": (PerturbationBC.choked_nozzle(), (c / rho) * M / (2.0 + gm1 * M),
                      (2.0 - gm1 * M) / (2.0 + gm1 * M)),
    "constant_mass_flow": (PerturbationBC.constant_mass_flow(), c * M / (rho * (1.0 - M)),
                           (1.0 + M) / (1.0 - M)),
}

print(f"{'closure':20s} {'meas R_s = g/h':>16s} {'analytic R_s':>16s}   {'R (acoustic)':>12s}")
for name, (bc, Rs_th, R_th) in couplers.items():
    fr, _ = terminate(bc, inlet_bc=PerturbationBC.anechoic(driven=("entropy",)))
    f_w, g_w, h_w = fr.waves(OUT)[0]
    Rs_meas = g_w / h_w
    print(f"{name:20s} {Rs_meas.real:>16.4f} {Rs_th:>16.4f}   {R_th:>12.4f}")
print("\n(compact_nozzle is an alias of choked_nozzle; at M -> 0 both R_s -> 0 and R -> +1, a hard wall.)")

## 4. `inherit` — keep the linearized mean boundary

The default. No reflection face is stamped; the terminal keeps the row produced by linearizing
its *mean* BC. Here the mean BC is a fixed static pressure, so its linearization is exactly
$p'=0$ — the ideal pressure release — and `inherit` reproduces `open_end` ($R=-1$) as a sanity
check. The point is that `inherit` tracks the mean condition rather than a prescribed $R$: it is
the right choice when the mean BC already encodes the acoustic physics you want.

In [ ]:
fr_inh, _ = terminate(PerturbationBC.inherit())  # == passing no perturbation_bc at all
fr_open, _ = terminate(PerturbationBC.open_end())
print(f"inherit (linearized fixed-p outlet):  g/f = {fr_inh.reflection_at(OUT)[0]:+.4f}")
print(f"open_end (ideal pressure release):    g/f = {fr_open.reflection_at(OUT)[0]:+.4f}")
print("\ninherit follows the fixed static-pressure mean BC, which linearizes to p' = 0 -> R = -1 here.")

## Summary

* **Diagonal reflections** — `hard_wall`, `open_end`, `mean_flow_open_end`, `anechoic`,
  `reflection`, `impedance`/`impedance_polar` — each stamp a single $R$, recovered exactly as
  $g/f$ at the termination (§1).
* **`excitation`** supplies the forcing $b$ (the source) and an optional `base_R`; `family`
  selects an acoustic or entropy incoming wave (§2).
* **`choked_nozzle`/`compact_nozzle` and `constant_mass_flow`** are outlet closures with the
  entropy→acoustic coupling $R_s$ — indirect noise — verified against Marble–Candel (§3).
* **`inherit`** (default) keeps the linearized mean BC untouched (§4).
* Frequencies are in **Hz** throughout (project convention).